# Task 5: Strategy Backtesting

**GMF Investments — Optimal Portfolio vs 60/40 Benchmark**

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import json, os, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded!')

## 1. Load Data & Optimal Weights

In [ ]:
# Load prices
csv_path = '../data/processed/prices.csv'
if os.path.exists(csv_path):
    prices = pd.read_csv(csv_path, index_col=0, parse_dates=True)
else:
    raw = yf.download(['TSLA','BND','SPY'], start='2015-01-01', end='2026-06-30', auto_adjust=True)
    prices = raw['Close'].ffill().bfill()

# Load optimal weights
weights_path = '../data/processed/optimal_weights.json'
if os.path.exists(weights_path):
    with open(weights_path) as f:
        opt = json.load(f)
    opt_weights = np.array([opt['TSLA'], opt['BND'], opt['SPY']])
else:
    # Fallback weights if Task 4 not run
    opt_weights = np.array([0.6, 0.1, 0.3])

print(f'Optimal weights: TSLA={opt_weights[0]*100:.1f}%, BND={opt_weights[1]*100:.1f}%, SPY={opt_weights[2]*100:.1f}%')

# Backtest period: 2025 onwards (unseen by models)
returns = prices.pct_change().dropna()
backtest_returns = returns[returns.index >= '2025-01-01'][['TSLA','BND','SPY']]
print(f'Backtest period: {backtest_returns.index[0].date()} to {backtest_returns.index[-1].date()}')
print(f'Backtest days: {len(backtest_returns):,}')

## 2. Simulate Portfolios

In [ ]:
# Strategy: optimal weights (hold, no rebalancing)
strategy_daily = backtest_returns @ opt_weights
strategy_cum = (1 + strategy_daily).cumprod()

# Benchmark: 60% SPY / 40% BND
benchmark_weights = np.array([0.0, 0.4, 0.6])  # [TSLA, BND, SPY]
benchmark_daily = backtest_returns @ benchmark_weights
benchmark_cum = (1 + benchmark_daily).cumprod()

print(f'Strategy final cumulative return: {(strategy_cum.iloc[-1]-1)*100:.2f}%')
print(f'Benchmark final cumulative return: {(benchmark_cum.iloc[-1]-1)*100:.2f}%')

## 3. Performance Metrics

In [ ]:
RISK_FREE_DAILY = 0.05 / 252

def portfolio_metrics(daily_returns, cum_returns, name):
    total_return = cum_returns.iloc[-1] - 1
    n_days = len(daily_returns)
    ann_return = (1 + total_return)**(252/n_days) - 1
    ann_vol = daily_returns.std() * np.sqrt(252)
    sharpe = (daily_returns.mean() - RISK_FREE_DAILY) / daily_returns.std() * np.sqrt(252)
    # Max drawdown
    rolling_max = cum_returns.cummax()
    drawdown = (cum_returns - rolling_max) / rolling_max
    max_dd = drawdown.min()
    print(f'\n=== {name} ===')
    print(f'  Total Return:       {total_return*100:+.2f}%')
    print(f'  Annualized Return:  {ann_return*100:+.2f}%')
    print(f'  Annualized Vol:     {ann_vol*100:.2f}%')
    print(f'  Sharpe Ratio:       {sharpe:.3f}')
    print(f'  Max Drawdown:       {max_dd*100:.2f}%')
    return {'name':name,'total':total_return,'ann_return':ann_return,
            'vol':ann_vol,'sharpe':sharpe,'max_dd':max_dd}

strat_m = portfolio_metrics(strategy_daily, strategy_cum, 'GMF Optimal Strategy')
bench_m = portfolio_metrics(benchmark_daily, benchmark_cum, '60/40 Benchmark')

## 4. Visualize Cumulative Returns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Cumulative returns
axes[0].plot(strategy_cum.index, (strategy_cum-1)*100,
             color='#e74c3c', linewidth=2, label='GMF Optimal Strategy')
axes[0].plot(benchmark_cum.index, (benchmark_cum-1)*100,
             color='steelblue', linewidth=2, linestyle='--', label='60/40 Benchmark')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle=':')
axes[0].fill_between(strategy_cum.index,
                      (strategy_cum-1)*100, (benchmark_cum-1)*100,
                      where=(strategy_cum > benchmark_cum),
                      alpha=0.15, color='green', label='Strategy outperforms')
axes[0].fill_between(strategy_cum.index,
                      (strategy_cum-1)*100, (benchmark_cum-1)*100,
                      where=(strategy_cum <= benchmark_cum),
                      alpha=0.15, color='red', label='Benchmark outperforms')
axes[0].set_title('Cumulative Returns: GMF Strategy vs 60/40 Benchmark (2025–2026)',
                   fontweight='bold', fontsize=13)
axes[0].set_ylabel('Cumulative Return (%)')
axes[0].legend(fontsize=10)

# Drawdown
roll_max_s = strategy_cum.cummax()
dd_s = (strategy_cum - roll_max_s) / roll_max_s * 100
roll_max_b = benchmark_cum.cummax()
dd_b = (benchmark_cum - roll_max_b) / roll_max_b * 100

axes[1].fill_between(dd_s.index, dd_s, 0, alpha=0.4, color='#e74c3c', label='Strategy Drawdown')
axes[1].fill_between(dd_b.index, dd_b, 0, alpha=0.4, color='steelblue', label='Benchmark Drawdown')
axes[1].plot(dd_s.index, dd_s, color='#e74c3c', linewidth=1)
axes[1].plot(dd_b.index, dd_b, color='steelblue', linewidth=1, linestyle='--')
axes[1].set_title('Drawdown Analysis', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Backtest plot saved!')

In [ ]:
# Performance comparison table
comp = pd.DataFrame([strat_m, bench_m]).set_index('name')
comp_display = pd.DataFrame({
    'Total Return': [f"{strat_m['total']*100:+.2f}%", f"{bench_m['total']*100:+.2f}%"],
    'Ann. Return':  [f"{strat_m['ann_return']*100:+.2f}%", f"{bench_m['ann_return']*100:+.2f}%"],
    'Ann. Vol':     [f"{strat_m['vol']*100:.2f}%", f"{bench_m['vol']*100:.2f}%"],
    'Sharpe':       [f"{strat_m['sharpe']:.3f}", f"{bench_m['sharpe']:.3f}"],
    'Max Drawdown': [f"{strat_m['max_dd']*100:.2f}%", f"{bench_m['max_dd']*100:.2f}%"],
}, index=['GMF Strategy', '60/40 Benchmark'])

print('=== BACKTEST PERFORMANCE COMPARISON ===')
print(comp_display.to_string())
comp_display.to_csv('../data/processed/backtest_metrics.csv')
print('\nSaved backtest_metrics.csv')

## 5. Conclusion

### Strategy Viability Assessment

The GMF Optimal Strategy (Maximum Sharpe Ratio portfolio) was backtested against the 60/40 SPY/BND benchmark using 2025–2026 data that was NOT used in model training.

**Key findings:**
- The strategy's performance is driven primarily by TSLA's weight, which experienced extreme volatility in 2025 (dropping from ~$420 to ~$220 before recovering to ~$490)
- Higher TSLA allocation increases both upside potential and maximum drawdown significantly
- The 60/40 benchmark provides much lower volatility and drawdown, at the cost of lower potential return

**Limitations of this backtest:**
1. **Short window:** 18 months is insufficient to distinguish skill from luck
2. **No transaction costs:** Real trading incurs fees, spreads, and slippage
3. **No rebalancing:** Monthly rebalancing back to target weights would reduce drift risk
4. **Single backtest:** One historical period may not represent future market regimes
5. **TSLA concentration risk:** Heavy TSLA weighting creates idiosyncratic risk

**Recommendation:** For conservative GMF clients, the Minimum Volatility portfolio is preferred. For growth-oriented clients with 5+ year horizons, the Maximum Sharpe portfolio remains appropriate.